# VM über eine Client API erstellen

Um mit der Google Cloud Platform (GCP) zu interagieren, gibt es drei gängige Wege:
1. **Google Cloud Console:** Die grafische Benutzeroberfläche (Webbrowser).
2. **gcloud CLI:** Das Kommandozeilenwerkzeug für das Terminal.
3. **Cloud Client Libraries:** Programmierschnittstellen (SDKs) für Sprachen wie Python, Go oder Java.

Während die Console gut für manuelle Aufgaben ist, ermöglichen Client Libraries die **Automatisierung** und das Konzept von **Infrastructure as Code (IaC)**. 
Die Python Client Libraries von Google sind modern, idiomatisch und verbergen die Komplexität der zugrunde liegenden REST- oder gRPC-Schnittstellen. In dieser Übung nutzen wir die Bibliothek `google-cloud-compute`, um eine virtuelle Maschine (Compute Engine) programmatisch zu verwalten.

## Vorbereitung der Umgebung
Bevor wir Code ausführen können, müssen wir die notwendigen Bibliotheken in unsere Python-Umgebung installieren.
Das Paket `google-cloud-compute` enthält alle Klassen, um Ressourcen wie Instanzen, Festplatten und Netzwerke in der Compute Engine zu steuern.
Wir können es einfach über `pip` installieren

In [ ]:
import sys
# Installation der Google Cloud Compute Library
!{sys.executable} -m pip install google-cloud-compute

## Authentifizierung
Jeder Zugriff auf die Cloud muss autorisiert sein. Für die Aufgabe bieten sich hierfür entweder ein *Access Token* oder ein *Service Account Key* an

* **Access Token:** Ein temporäres Token (Zeichenkette beginnend mit `ya29...`), das standardmäßig nur 60 Minuten gültig ist und sich damit gut für kurze Tests (wie in unserer Aufgabe) eignet. Ein Access Token wird von einer Identität (z. B. Benutzer oder Dienstkonto) bezogen und erlaubt Zugriff entsprechend deren Berechtigungen.
* **Service Account Key (JSON):** Eine Datei, die dauerhafte Anmeldedaten für einen *Service-Account* enthält. Wer den Schlüssel besitzt, hat damit alle Berechtigungen des Service-Accounts im Projekt.

In dieser Aufgabe nutzen wir ein manuell bereitgestelltes Access Token. Beachte, dass dieses Token abläuft und bei Fehlermeldungen (z.B. `401 Unauthorized`) erneuert werden muss.

Um ein Access Token zu beziehen, rufen Sie in der Console folgenden `gcloud` Befel auf:
```bash
gcloud auth print-access-token
````

In [ ]:
# Fügen Sie hier Ihr Access Token ein, das Sie lokal über 'gcloud auth print-access-token' generiert haben.
access_token = 'HIER_TOKEN_EINFÜGEN'

## Konfiguration der VM
Um eine VM zu erstellen, müssen wir festlegen, *wo* sie laufen soll und *was* sie beinhalten soll.

Wir nutzen die Zone `europe-west3-{a,b,c}` (Frankfurt) und den kleinstmöglichen Maschinentyp `e2-micro`, um die Kosten gering zu halten.

In der GCP Client API bauen wir Ressourcen modular zusammen. Eine `Instance` besteht aus:
* Einem **Machine Type** (CPU/RAM).
* Einem **Boot Disk** (Das Betriebssystem, hier Debian 12).
* Einem **Network Interface** (Die Verbindung zum Internet über das 'default' Netzwerk).


In [ ]:
from google.oauth2.credentials import Credentials
from google.cloud import compute_v1
import random

# Credentials-Objekt erstellen
# Die Library google-auth nutzt dieses Objekt, um das Token bei jedem Request im Header mitzusenden.
creds = Credentials(access_token)
instance_client = compute_v1.InstancesClient(credentials=creds)

# Projekt-Spezifikationen
project_id = "cloud-computing-ss26"
zone = f"europe-west3-{random.choice(['a', 'b', 'c'])}"

# VM-Konfiguration
instance_name = "mein-name-vm-1" # Benennen Sie die VM nach Ihrem Schema
instance = compute_v1.Instance()
instance.name = instance_name
instance.machine_type = f"zones/{zone}/machineTypes/e2-micro"

# Festplatte konfigurieren
initialize_params = compute_v1.AttachedDiskInitializeParams()
initialize_params.source_image = "projects/debian-cloud/global/images/family/debian-12"
initialize_params.disk_size_gb = 10

boot_disk = compute_v1.AttachedDisk()
boot_disk.initialize_params = initialize_params
boot_disk.auto_delete = True # Löscht die Disk, wenn die VM gelöscht wird
boot_disk.boot = True
instance.disks = [boot_disk]

# Netzwerk-Schnittstelle (Standard-VPC)
network_interface = compute_v1.NetworkInterface()
network_interface.name = "global/networks/default"
instance.network_interfaces = [network_interface]

## Erzeugung der VM 

Cloud-Operationen (wie das Starten einer VM) passieren nicht sofort. Der API-Aufruf `insert` startet den Prozess und gibt ein `Operation`-Objekt zurück.
Wir nutzen den blockierenden Aufruf `operation.result()` um im Programmcode solange zu warten, bis die Google Cloud meldet, dass die VM fertig hochgefahren ist.

In [ ]:
print(f"Starte VM '{instance_name}'...")

operation = instance_client.insert(
    project=project_id,
    zone=zone,
    instance_resource=instance
)

operation.result() 

print("Die VM wurde erstellt.")

## Cleanup

Überprüfen Sie über die Cloud-Console, dass die VM läuft und löschen Sie danach direkt die erstellte VM wieder über die API.

In [ ]:
print(f"Löschvorgang für VM '{instance_name}' wird eingeleitet...")

operation = instance_client.delete(
    project=project_id,
    zone=zone,
    instance=instance_name
)

operation.result()

print(f"VM '{instance_name}' wurde erfolgreich entfernt.")